# Variational optimization loop

Run the same fixed-step gradient-descent loop with a reference QNode and the MettleQ device.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

A hybrid loop repeatedly evaluates a QNode, computes gradients, and updates classical parameters.

In [2]:
def make_qnode(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def circuit(weights):
        qml.RY(weights[0], wires=0)
        qml.RX(weights[1], wires=1)
        qml.CNOT(wires=[0, 1])
        qml.RY(weights[2], wires=1)
        return qml.expval(qml.Z(0) @ qml.Z(1))
    return circuit

def train(qnode):
    weights = pnp.array([0.2, -0.4, 0.7], requires_grad=True)
    trace = []
    for _ in range(8):
        value = qnode(weights)
        trace.append(float(value))
        weights = weights - 0.15 * qml.grad(qnode)(weights)
    trace.append(float(qnode(weights)))
    return np.asarray(trace)

reference_qnode = make_qnode(qml.device("default.qubit", wires=2))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: train(reference_qnode), repeats=2)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: train(mettleq_qnode), repeats=2)
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

The full optimization loss trace is compared to catch divergence at any step.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/04_variational_optimization.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="optimization trace atol=4e-5",
    passed=error <= 4e-5 and candidate[-1] <= candidate[0],
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_trace_error": error, "reference_trace": reference, "mettleq_trace": candidate},
)


Comparison summary
------------------
Correctness contract: PASS — optimization trace atol=4e-5
SDK reference median: 46.591 ms
MettleQ median:       99.578 ms
Timing interpretation: the SDK reference was 2.137x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "optimization trace atol=4e-5", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"max_trace_error": 9.725507865709915e-08, "mettleq_trace": [0.7044664025306702, 0.6360342502593994, 0.5586625933647156, 0.47383859753608704, 0.38371843099594116, 0.2907371520996094, 0.19708256423473358, 0.10422448813915253, 0.012694669887423515], "reference_trace": [0.7044663052755915, 0.6360342844556788, 0.5586625801001408, 0.4738386056809144, 0.38371844500894703, 0.2907372202985556, 0.19708257496945103, 0.10422440007088346, 0.012694577493989834]}, "met

## What should you conclude?

Use MettleQ when quantum evaluations dominate the optimizer; two-wire training is mostly Python overhead.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.